In [38]:
from git import Repo
from langchain.text_splitter import Language, RecursiveCharacterTextSplitter
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain.memory import ConversationSummaryMemory
from langchain.chains import ConversationalRetrievalChain
import os

## Clone the Git hub repo here


In [39]:
os.makedirs("test_repo", exist_ok=True)

In [40]:
repo_path = "test_repo"
if not os.path.exists(repo_path) or not os.listdir(repo_path):
    repo = Repo.clone_from("https://github.com/entbappy/End-to-End-Medical-Chatbot", to_path=repo_path)
else:
    repo = Repo(repo_path)
    print("Repository already available at", repo_path)

Repository already available at test_repo


## Load the data from the cloned folders

In [23]:
%pwd

'd:\\Real Time Source code analyzer\\research'

In [41]:
loader = GenericLoader.from_filesystem(repo_path,
                                      glob="**/*.*",
                                      suffixes=[".py"],
                                      parser=LanguageParser(language=Language.PYTHON, parser_threshold=500)
 )

In [42]:
documents = loader.load()

In [26]:
documents 

[Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='from flask import Flask, render_template, jsonify, request\nfrom src.helper import download_hugging_face_embeddings\nfrom langchain_pinecone import PineconeVectorStore\nfrom langchain_openai import ChatOpenAI\nfrom langchain.chains import create_retrieval_chain\nfrom langchain.chains.combine_documents import create_stuff_documents_chain\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom dotenv import load_dotenv\nfrom src.prompt import *\nimport os\n\napp = Flask(__name__)\n\n\nload_dotenv()\n\n\nPINECONE_API_KEY=os.environ.get(\'PINECONE_API_KEY\')\nOPENAI_API_KEY=os.environ.get(\'OPENAI_API_KEY\')\n\nos.environ["PINECONE_API_KEY"] = PINECONE_API_KEY\nos.environ["OPENAI_API_KEY"] = OPENAI_API_KEY\n\nembeddings = download_hugging_face_embeddings()\n\n\nindex_name = "medical-chatbot" \n# Embed each chunk and upsert the embeddings into your Pinecone index.\ndocsearch = Pin

In [27]:
len(documents)

6

## split the documents into chunks


In [43]:
document_spliter = RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON,
                                                                chunk_size=500, 
                                                                chunk_overlap=20)

In [44]:
text = document_spliter.split_documents(documents)

In [30]:
text

[Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='from flask import Flask, render_template, jsonify, request\nfrom src.helper import download_hugging_face_embeddings\nfrom langchain_pinecone import PineconeVectorStore\nfrom langchain_openai import ChatOpenAI\nfrom langchain.chains import create_retrieval_chain\nfrom langchain.chains.combine_documents import create_stuff_documents_chain\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom dotenv import load_dotenv\nfrom src.prompt import *\nimport os\n\napp = Flask(__name__)\n\n\nload_dotenv()'),
 Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='load_dotenv()\n\n\nPINECONE_API_KEY=os.environ.get(\'PINECONE_API_KEY\')\nOPENAI_API_KEY=os.environ.get(\'OPENAI_API_KEY\')\n\nos.environ["PINECONE_API_KEY"] = PINECONE_API_KEY\nos.environ["OPENAI_API_KEY"] = OPENAI_API_KEY\n\nembeddings = download_hugging_face_embeddings()\n

In [31]:
len(text)

14

## Configure Groq API key

In [45]:
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("Missing GROQ_API_KEY in environment or .env file")

In [46]:
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

## Set up vector database (local embeddings + Chroma)

In [48]:
%pip install -q sentence-transformers chromadb

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.


In [49]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [50]:
vectordb = Chroma.from_documents(text,embedding=embeddings, persist_directory="./data")

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k": 4})

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=GROQ_API_KEY,
 )

memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    return_messages=True
 )

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectordb.as_retriever(search_kwargs={"k": 8}),
    memory=memory
 )

print("Groq-based RAG chain is ready.")

Groq-based RAG chain is ready.


C:\Users\saiki\AppData\Local\Temp\ipykernel_30168\4260297288.py:9: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryMemory(


In [52]:
vectordb.persist()

C:\Users\saiki\AppData\Local\Temp\ipykernel_30168\3711397106.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


## Question anad Answer

In [54]:
question = "what is download_data function in data_loader.py file?"

In [55]:
result = qa_chain.run(question)

C:\Users\saiki\AppData\Local\Temp\ipykernel_30168\3726660460.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain.run(question)
